# Native Plant LLM Enrichment Pipeline

Enriches raw wildflower.org plant records with LLM-generated botanical attributes (dimensions, phenology, propagation, rarity, etc.) using a two-pass chain: **Claude Sonnet 4.5** for full enrichment, **Llama 4 Maverick** to validate uncertain fields.

**Source:** `powers_land_catalog.natives.raw_wildflower_org_plants` (2,046 unique plants after de-dupe)  
**Target:** `powers_land_catalog.natives.enriched_wildflower_org_plants`  
**Avg cost:** ~$0.0087/plant · ~$17.80 for full run  
**Avg runtime:** ~18-20 min (10 parallel workers)


## Schema

```
avg_spread_ft: decimal
avg_height_ft: decimal
form: ENUM(climbing, clump, mound, upright, rosette, shrub, spreading)
foliage_type: ENUM(sword, blade, broadleaf, fine)
leaf_width: ENUM(narrow, medium, wide)
foliage_color: ENUM(X11 color list)
flower_color: ENUM(X11 color list)
lifespan: ENUM(annual, biennial, perennial, evergreen)
deciduous_ind: ENUM(E for Evergreen, SE for semievergreen, D for Deciduous)
plant_hardiness_zone: ENUM(example: 7a, 8b)
native_region: ENUM(
    E - Edwards Plateau: Shallow limestone or caliche soil
    ,B - Blackland Prairie: Deeper, dark clay soils 
    ,B/E - Native to both Blackland Prairie and Edwards Plateau
    ,T - Texas (outside of the immediate Austin area)
    ,X - Not native but adapted to Texas
)
water_requirements: ENUM(
    VL: Very Low (Water occasionally during very dry conditions)
    ,L: Low (Water thoroughly every 3-4 weeks if no rainfall)
    ,M: Medium (Water thoroughly every 2-3 weeks if no rainfall)
    ,H: High (Water thoroughly every 5-7 days if no rainfall)
)
light_requirement: ENUM(sun, part-sun, part-shade, shade)
bloom_months: multi select array for month number - [5,6,7,8]
propagation: string (notes on how the plant spreads, via cuttings or seeds or rhyzomes, etc)
notes: string (tips, tricks, or good uses on the plant. Anything that is good know. Should be very specific. If there is nothing to offer, show NULL)
rarity: integer (plant rarity, how often it can be found in nurseries, plant stores, etc. Scale of 1-5)
problems: String (any notes on the pests, fungus, etc)
confidence: decimal (confidence rating for the LLM to know specifics on the plant)
```

## plant rarity scale: 
| Score | Label | Wild status (native habitat) | Cultivation/availability |
|---|---|---|---|
| 1 | Ubiquitous | Widespread, abundant across a large range | Stocked by most nurseries |
| 2 | Common | Widespread, but patchier or habitat-dependent | Easy to find at specialty or regional nurseries |
| 3 | Uncommon | Limited range or narrow habitat, but stable populations | A handful of specialist growers |
| 4 | Rare | Small range and/or small populations, often a conservation concern | Few growers, often by collectors or botanic gardens |
| 5 | Exceptional | Critically scarce, endemic to a tiny area, or known from few individuals | Held by a very small number of people or institutions |


In [0]:
%uv pip install openai

In [0]:
from pyspark.sql import functions as F

# ── Read source table ─────────────────────────────────────────────────────────
df_plants = spark.table("powers_land_catalog.natives.raw_wildflower_org_plants") \
    .filter(F.col("scientific_name").isNotNull())

# ── De-dupe on scientific_name, keeping first occurrence per plant ────────────
df_deduped = df_plants.dropDuplicates(["scientific_name"])

print(f"Rows before de-dupe : {df_plants.count()}")
print(f"Unique plants        : {df_deduped.count()}")
display(df_deduped)

In [0]:
SYSTEM_PROMPT = (
    "You are a professional botanist and horticulturalist with deep expertise in Central Texas / Austin area native plants. "
    "Respond only with valid JSON matching the schema provided. "
    "Use null for any field you cannot determine with reasonable confidence."
)


def build_plant_prompt(row) -> str:
    return f"""Provide structured botanical data for the following plant.

Known information:
  Common name      : {row['common_name']}
  Botanical name   : {row['scientific_name']}
  Duration         : {row['Duration']}
  Habit            : {row['Habit']}
  Sun              : {row['Sun']}
  Water            : {row['Water']}
  Source list      : {row['source_list']}

Return ONLY a JSON object with these fields:

{{
  "avg_spread_ft"        : <decimal>,
  "avg_height_ft"        : <decimal>,
  "form"                 : <climbing | clump | mound | upright | rosette | shrub | spreading>,
  "foliage_type"         : <sword | blade | broadleaf | fine>,
  "leaf_width"           : <narrow | medium | wide>,
  "foliage_color"        : <X11 color name — lowercase, no spaces, dominant color only (e.g. "darkgreen", "sienna")>,
  "flower_color"         : <X11 color name — lowercase, no spaces, dominant color only (e.g. "violet", "white")>,
  "lifespan"             : <annual | biennial | perennial>,
  "deciduous_ind"        : <E | SE | D>,
  "plant_hardiness_zone" : <e.g. "7a" or "7a-9b">,
  "native_region"        : <E | B | B/E | T | X>,
  "water_requirements"   : <VL | L | M | H>,
  "light_requirement"    : <sun | part-sun | part-shade | shade>,
  "bloom_months"         : <array of month numbers, e.g. [3, 4, 5]>,
  "propagation"          : <array of one or more: seed | division | cuttings | bulb_offsets | rhizomes | runners | layering | spores | suckering>,
  "notes"                : <string max 200 chars, or null>,
  "rarity"               : <integer 1-5>,
  "problems"             : <string max 200 chars, or null>,
  "low_confidence_fields": <array of field names where you are not confident in the value, e.g. ["bloom_months", "rarity"]>
}}

Reference legends:

avg_spread_ft:
  Provided as fractional feet. A plant that grows between 6-12 inches wide should show .75
  
avg_height_ft: 
  Provided as fractional feet. A plant that grows between 6-12 inches tall should show .75

color fields (foliage_color, flower_color):
  Use exact X11 color names — lowercase, no spaces (e.g. "darkgreen", "violet").
  When multiple colors are present, use the single dominant color.
  For near-white or pale tones, use the closest X11 match (e.g. "white", "lightyellow", "lavenderblush").

deciduous_ind:
  E   = Evergreen 
  SE  = Semi-Evergreen (dies back during periods or low temps or high heat)
  D   = Deciduous

native_region (Central Texas / Austin area context):
  E   = Edwards Plateau (shallow limestone / caliche soil)
  B   = Blackland Prairie (deep, dark clay soils)
  B/E = Native to both Blackland Prairie and Edwards Plateau
  T   = Texas (outside the immediate Austin area)
  X   = Not native but adapted to Texas

water_requirements:
  VL = Very Low  — water occasionally during very dry conditions
  L  = Low       — water thoroughly every 3-4 weeks if no rainfall
  M  = Medium    — water thoroughly every 2-3 weeks if no rainfall
  H  = High      — water thoroughly every 5-7 days if no rainfall

rarity scale:
  1 = Ubiquitous  — stocked by most nurseries
  2 = Common      — easy to find at specialty / regional nurseries
  3 = Uncommon    — a handful of specialist growers
  4 = Rare        — few growers; collectors or botanic gardens
  5 = Exceptional — held by very few people or institutions

Example output (Allium drummondii — Drummond's Onion):
{{
  "avg_spread_ft"        : 0.5,
  "avg_height_ft"        : 0.75,
  "form"                 : "clump",
  "foliage_type"         : "blade",
  "leaf_width"           : "narrow",
  "foliage_color"        : "green",
  "flower_color"         : "white",
  "lifespan"             : "perennial",
  "deciduous_ind"        : "D",
  "plant_hardiness_zone" : "6b-9a",
  "native_region"        : "E",
  "water_requirements"   : "VL",
  "light_requirement"    : "sun",
  "bloom_months"         : [3, 4, 5],
  "propagation"          : ["seed", "bulb_offsets", "division"],
  "notes"                : "Edible bulbs and leaves with mild onion flavor. Naturalizes well in rocky limestone soils. Attractive to bees.",
  "rarity"               : 2,
  "problems"             : null,
  "low_confidence_fields": ["plant_hardiness_zone"]
}}
"""


# ── Smoke-test on first plant ─────────────────────────────────────────────────
sample = df_deduped.limit(1).collect()[0]
print("=== SYSTEM ===")
print(SYSTEM_PROMPT)
print("\n=== USER ===")
print(build_plant_prompt(sample))

In [0]:
import ast, json, re, time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from databricks.sdk import WorkspaceClient

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_PASS1 = "system.ai.claude-sonnet-4-5"   # primary enrichment
MODEL_PASS2 = "system.ai.llama-4-maverick"    # uncertainty validator
MAX_WORKERS = 10                              # concurrent requests (~18-20 min for 2k plants)
MAX_RETRIES = 3                               # retries on rate-limit errors

# ── Client ────────────────────────────────────────────────────────────────────
_w  = WorkspaceClient()
_tk = _w.config.authenticate().get("Authorization", "").split(" ", 1)[-1]
_client = OpenAI(api_key=_tk, base_url=f"{_w.config.host}/ai-gateway/mlflow/v1")

# ── Field definitions (used in validation prompt) ─────────────────────────────
FIELD_DEFS = {
    "avg_spread_ft"        : "decimal feet (e.g. 0.75 = 6-9 in)",
    "avg_height_ft"        : "decimal feet (e.g. 0.75 = 6-9 in)",
    "form"                 : "climbing | clump | mound | upright | rosette | shrub | spreading",
    "foliage_type"         : "sword | blade | broadleaf | fine",
    "leaf_width"           : "narrow | medium | wide",
    "foliage_color"        : "X11 color name — lowercase, no spaces",
    "flower_color"         : "X11 color name — lowercase, no spaces",
    "lifespan"             : "annual | biennial | perennial",
    "deciduous_ind"        : "E (Evergreen) | SE (Semi-Evergreen) | D (Deciduous)",
    "plant_hardiness_zone" : 'e.g. "7a" or "7b-9a"',
    "native_region"        : "E (Edwards Plateau) | B (Blackland Prairie) | B/E (both) | T (Texas outside Austin) | X (adapted)",
    "water_requirements"   : "VL (very low) | L (low) | M (medium) | H (high)",
    "light_requirement"    : "sun | part-sun | part-shade | shade",
    "bloom_months"         : "array of month numbers e.g. [3, 4, 5]",
    "propagation"          : "array: seed | division | cuttings | bulb_offsets | rhizomes | runners | layering | spores | suckering",
    "notes"                : "string max 200 chars or null",
    "rarity"               : "1=ubiquitous at nurseries … 5=extremely rare",
    "problems"             : "string max 200 chars or null",
}


# ── Helpers ───────────────────────────────────────────────────────────────────

def call_llm(model, user_prompt, temperature=0):
    """Returns (raw | None, elapsed_s, err | None, usage | None). Retries on 429."""
    for attempt in range(MAX_RETRIES):
        t0 = time.time()
        try:
            kwargs = dict(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_prompt},
                ],
                max_tokens=1024,
            )
            if temperature is not None:
                kwargs["temperature"] = temperature
            resp = _client.chat.completions.create(**kwargs)
            raw = resp.choices[0].message.content
            if isinstance(raw, list):
                raw = "".join(b["text"] if isinstance(b, dict) and "text" in b else str(b) for b in raw)
            usage = {"prompt_tokens": resp.usage.prompt_tokens, "completion_tokens": resp.usage.completion_tokens}
            return raw, round(time.time() - t0, 2), None, usage
        except Exception as e:
            elapsed = round(time.time() - t0, 2)
            err_str = str(e)
            if ("429" in err_str or "rate_limit" in err_str.lower()) and attempt < MAX_RETRIES - 1:
                time.sleep(2 ** attempt)   # 1 s, 2 s back-off
                continue
            return None, elapsed, err_str, None
    return None, 0.0, f"Rate-limited after {MAX_RETRIES} retries", None


def parse_json_safe(raw):
    """Strip markdown fences and parse JSON. Returns (dict | None, err | None)."""
    if raw is None:
        return None, "no response"
    clean = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.IGNORECASE)
    clean = re.sub(r"\s*```$", "", clean.strip())
    try:
        return json.loads(clean), None
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(clean), None
        except Exception as e:
            return None, f"{e} | raw[:150]: {raw[:150]!r}"


def build_validation_prompt(row, pass1_data, uncertain_fields):
    """Second-pass prompt asking Llama to review only the uncertain fields."""
    lines = [
        f'  "{f}": {json.dumps(pass1_data.get(f))}  // {FIELD_DEFS.get(f, "see schema")}'
        for f in uncertain_fields
    ]
    return f"""You are a professional botanist reviewing uncertain fields in a plant database.

Plant: {row['scientific_name']} ({row['common_name']})
Duration: {row['Duration']} | Habit: {row['Habit']} | Sun: {row['Sun']} | Water: {row['Water']}

The primary model was uncertain about these fields (current values shown):
{{
{chr(10).join(lines)}
}}

Return ONLY a JSON object containing ONLY these {len(uncertain_fields)} fields plus "still_uncertain":
  Fields to review: {json.dumps(uncertain_fields)}
  "still_uncertain": array of any field names you also cannot confirm (can be [])

Correct any value you are confident about. Leave unchanged any value you agree with or cannot improve.
Return valid JSON only. No explanation.
"""


def enrich_plant(row):
    """Run the two-pass enrichment chain for one plant. Thread-safe."""
    name = row["scientific_name"]

    # Pass 1: Sonnet 4.5 — full enrichment
    raw1, lat1, err1, tok1 = call_llm(MODEL_PASS1, build_plant_prompt(row), temperature=0)
    p1, perr1 = parse_json_safe(raw1)

    if p1 is None:
        return {
            "scientific_name" : name, "common_name": row["common_name"],
            "pass1_ok"        : False, "pass2_ran": False,
            "pass1_latency_s" : lat1,  "pass2_latency_s": None,
            "uncertain_in"    : [],    "still_uncertain": [],
            "p1_in_tokens"    : tok1["prompt_tokens"] if tok1 else None,
            "p1_out_tokens"   : tok1["completion_tokens"] if tok1 else None,
            "p2_in_tokens"    : None,  "p2_out_tokens": None,
            "final_data"      : None,  "error": err1 or perr1,
        }

    uncertain = p1.get("low_confidence_fields", [])

    # Pass 2: Llama — validate uncertain fields only (skip if none)
    if uncertain:
        raw2, lat2, err2, tok2 = call_llm(MODEL_PASS2, build_validation_prompt(row, p1, uncertain), temperature=0)
        p2, perr2 = parse_json_safe(raw2)
        if p2:
            still_unc = p2.pop("still_uncertain", [])
            for f in uncertain:
                if f not in still_unc and f in p2 and p2[f] is not None:
                    p1[f] = p2[f]
            p1["still_uncertain_after_chain"] = still_unc
        else:
            still_unc = uncertain
            p1["still_uncertain_after_chain"] = still_unc
        pass2_ran, lat2_out, tok2_out = True, lat2, tok2
    else:
        still_unc, pass2_ran, lat2_out, tok2_out = [], False, None, None
        p1["still_uncertain_after_chain"] = []

    return {
        "scientific_name"  : name,
        "common_name"      : row["common_name"],
        "pass1_ok"         : True,
        "pass2_ran"        : pass2_ran,
        "pass1_latency_s"  : lat1,
        "pass2_latency_s"  : lat2_out,
        "uncertain_in"     : uncertain,
        "still_uncertain"  : p1.get("still_uncertain_after_chain", []),
        "p1_in_tokens"     : tok1["prompt_tokens"] if tok1 else None,
        "p1_out_tokens"    : tok1["completion_tokens"] if tok1 else None,
        "p2_in_tokens"     : tok2_out["prompt_tokens"] if tok2_out else None,
        "p2_out_tokens"    : tok2_out["completion_tokens"] if tok2_out else None,
        "final_data"       : p1,
        "error"            : None,
    }


# ── Full run ────────────────────────────────────────────────────────────────────
all_rows = df_deduped.collect()
n        = len(all_rows)
print(f"Processing {n} plants with {MAX_WORKERS} workers …")
t_start = time.time()

results = [None] * n
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_idx = {executor.submit(enrich_plant, row): i for i, row in enumerate(all_rows)}
    done = 0
    for future in as_completed(future_to_idx):
        results[future_to_idx[future]] = future.result()
        done += 1
        if done % 100 == 0 or done == n:
            elapsed = time.time() - t_start
            rate    = done / elapsed
            eta     = (n - done) / rate if rate else 0
            print(f"  {done:>4}/{n}  {elapsed/60:.1f} min elapsed  ETA {eta/60:.1f} min")

df_enriched = pd.DataFrame(results)

# ── Summary ───────────────────────────────────────────────────────────────────
ok       = int(df_enriched["pass1_ok"].sum())
p2_count = int(df_enriched["pass2_ran"].sum())
unc_in   = df_enriched["uncertain_in"].apply(len).sum()
unc_out  = df_enriched["still_uncertain"].apply(len).sum()
wall     = time.time() - t_start

print(f"\n{'\u2500'*55}")
print(f"Plants enriched   : {ok}/{n}  ({n - ok} failed)")
print(f"Pass 2 triggered  : {p2_count}/{n}")
print(f"Uncertain fields  : {unc_in} \u2192 {unc_out} remaining  ({unc_in - unc_out} resolved)")
print(f"Wall time         : {wall/60:.1f} min")
print(f"Est. cost         : ~${ok * 0.0087:.2f}  (@$0.0087/plant)")

In [0]:
import numpy as np  # noqa: F401

# ⚠️  Confirm TARGET_TABLE name before running this cell
TARGET_TABLE = "powers_land_catalog.natives.enriched_wildflower_org_plants"

ENRICH_FIELDS = [
    "avg_spread_ft", "avg_height_ft", "form", "foliage_type", "leaf_width",
    "foliage_color", "flower_color", "lifespan", "deciduous_ind",
    "plant_hardiness_zone", "native_region", "water_requirements",
    "light_requirement", "bloom_months", "propagation", "notes",
    "rarity", "problems",
]

records = []
for _, r in df_enriched.iterrows():
    if r["final_data"] is None:
        continue
    d = r["final_data"]
    records.append({
        "scientific_name"        : r["scientific_name"],
        "common_name"            : r["common_name"],
        **{f: d.get(f) for f in ENRICH_FIELDS},
        "still_uncertain_fields" : r["still_uncertain"] or [],
        "enrichment_pass2_ran"   : bool(r["pass2_ran"]),
        "p1_in_tokens"           : int(r["p1_in_tokens"]) if pd.notna(r["p1_in_tokens"]) else None,
        "p1_out_tokens"          : int(r["p1_out_tokens"]) if pd.notna(r["p1_out_tokens"]) else None,
        "p2_in_tokens"           : int(r["p2_in_tokens"]) if pd.notna(r["p2_in_tokens"]) else None,
        "p2_out_tokens"          : int(r["p2_out_tokens"]) if pd.notna(r["p2_out_tokens"]) else None
    })

# Go through pandas first to normalize mixed int/float types across rows
df_write = spark.createDataFrame(pd.DataFrame(records))
(
    df_write
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)
print(f"\u2713 Written {df_write.count()} rows to {TARGET_TABLE}")
display(spark.table(TARGET_TABLE).limit(5))